# 1-Implementar los endpoints

Haz una función en Python para cada uno de los endpoints del API REST:

- `GET /user/{email}`
- `GET /room/{room_id}`
- `GET /dungeon/{dungeon_id}`
- `POST /comment/`
- `DELETE /monster/{monster_id}`

Las funciones deben conectarse a la base de datos **MongoDB** y realizar las consultas pertinentes.

Se deben realizar todas las operaciones posibles directamente en la base de datos. No ejecutes cálculos en Python si no es necesario. 

In [26]:
from pymongo import MongoClient
from datetime import datetime

client = MongoClient("mongodb://localhost:27017/")
db = client["jotuns_lair"]

rooms    = db["rooms"]
users    = db["users"]
loot     = db["loot"]
monsters = db["monsters"]

## Endpoints a implementar

#### **Información de un usuario** 
````GET /user/{email} ````

Este endpoint recibe el **email** de un usuario y devuelve toda la información disponible del mismo.

Además, incluye los **20 últimos comentarios** que ha realizado ese usuario.  
De cada comentario se debe mostrar:

- **Texto**
- **Fecha de creación**
- **Categoría**
- **Id de la habitación** (`Room.IdR`)
- **Nombre de la habitación** (`Room.name`)
- **Id de la mazmorra** (`Dungeon.IdD`)
- **Nombre de la mazmorra** (`Dungeon.name`)

> Los comentarios deben ordenarse por fecha de creación, mostrando primero los más recientes.

In [27]:
def get_user(email: str):
    """
    GET /user/{email}
    Devuelve todos los campos del usuario más los 20 últimos comentarios
    con texto, fecha, categoría, id/nombre de room y id/nombre de dungeon.
    """
    pipeline = [
        # Filtrar por email
        {"$match": {"email": email}},

        # Ordenar los hints por fecha desc y quedarse con los 20 últimos 
        # TODO: preguntar si esto se aplica con lo que tenemos o con después de patrones de diseño
        {"$addFields": {
            "hints": {
                "$slice": [
                    {"$sortArray": {
                        "input": "$hints",
                        "sortBy": {"creation_date": -1}
                    }},
                    20
                ]
            }
        }},

        # Proyectar solo los campos necesarios
        {"$project": {
            "_id": 0,
            "email": 1,
            "user_name": 1,
            "creation_date": 1,
            "country": 1,
            "hints": {
                "text": 1,
                "creation_date": 1,
                "category": 1,
                "references_room.room_id": 1,
                "references_room.room_Name": 1,
                "references_room.dungeon_id": 1,
                "references_room.dungeon_name": 1
            }
        }}
    ]

    result = list(users.aggregate(pipeline))
    return result[0] if result else None

In [28]:
get_user("abbottanne@example.com")

{'email': 'abbottanne@example.com',
 'hints': [{'text': 'Identify than professor statement support campaign computer.\\nEvent part half use plan. Already development front. Need today today set cold rock husband go.\\nWant safe PM front. Although born speech other project decision.\\nQuickly account staff imagine unit interest pick. Modern cultural someone appear rich quickly science. Director three red nice. True manager TV somebody school practice he.\\nRate size air off. Certain difficult drop walk cold share.\\nDraw standard prevent whole appear seat stand.',
   'category': 'lore',
   'creation_date': '2017-12-01 12:52:27.000000'},
  {'text': 'President result month range set specific magazine natural. Much agency bed himself production east interview. Involve yard line hear. Eat bed behind put process.\\nMother discuss bank mouth media window focus conference. Avoid official owner evening news example somebody.\\nDrop from south only think. That fly soon understand require differe

#### **Información de habitación en particular** 
````GET /room/{room_id} ````

Este endpoint recibe el **id de una habitación** y devuelve la siguiente información:

- **`idR`**
- **`name`**
- **`inWP`**
- **`outWP`**

Además, incluye:

- El **número de monstruos de cada tipo** presentes en la habitación.
- El **total de oro** que valen los tesoros de la sala.
- Los **últimos 20 comentarios** realizados sobre esa habitación.

Cada comentario debe incluir:

- **`userName`**
- **`country`**
- **`creationDate`** del usuario que lo realizó
- **Texto**
- **Fecha de publicación**
- **Categoría** del comentario

#### **Información de una mazmorra** 
````GET /dungeon/{dungeon_id} ````

### Información de una mazmorra  
`GET /dungeon/{dungeon_id}`

Este endpoint recibe el identificador de una mazmorra y devuelve información clave de una mazmorra del juego.

Debe incluir

- **Datos principales de la mazmorra**:
    - `idM`
    - `name`
    - `lore`

- **Datos para el grafo interactivo**:
    1. **Habitaciones de la mazmorra**: `id` y `name` de cada habitación.
    2. **Conexiones entre habitaciones** dentro de la mazmorra.
    3. **Monstruos por habitación**: `id` y `name` de cada monstruo.
    4. **Tesoros por habitación**: `id` y `name` de cada tesoro.
    5. **Comentarios por categoría en cada habitación**: número total por categoría.

#### **Publicar comentario** 
````POST /comment/```` 

Este endpoint añade un nuevo comentario. Recibe como parámetros: user_email (str), room_id (int), 
text (str), category (str).

#### **Borrar monstruo** 
````DELETE /monsters/{monster_id}```` 

Este endpoint recibe el id de un monstruo y lo elimina de la base de datos.